# Raw Data Analysis Workflow

This notebook runs the full raw-data analysis (DLTS-per-pulse, TOF, M/C, FDM,
multi-hit / dead-zone) against a single PyCCAPT `.h5` file. Two file layouts
are accepted automatically:

- **Calibrated bundle** — the output of the data-processing notebook with
  `save_tdc=True` and `save_range=True`: `/df` (calibrated dld) plus
  `/tdc` (raw, linked) and optionally `/range`.
- **Pure raw acquisition** — the file as written by the control software,
  with `/dld` and `/tdc` groups and no calibrated `/df`.

The detector kind (Surface Concept vs RoentDek) is auto-detected from the
linked `/tdc` group. A single dropdown lets you choose whether the species
list comes from the loaded `/range` table or from manually typed peak
windows.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import subprocess
import warnings

import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore")

from pyccapt.calibration.core import share_variables
from pyccapt.calibration.tutorials.tutorials_helpers import (
    helper_auto_raw_analysis,
    helper_data_loader,
)

variables = share_variables.Variables()

## 1. Pick the `.h5` file

The selected file can be either a calibrated bundle (`/df` + `/tdc` +
optionally `/range`) or a pure raw acquisition file (`/dld` + `/tdc`).

In [ ]:
button = widgets.Button(description='Load dataset')

@button.on_click
def open_file_on_click(_):
    global dataset_path
    folder_path = variables.last_directory
    script = '..//..//data_tools//run_dataset_path_qt.py'
    result = subprocess.run(
        ['python', script, folder_path, 'dataset'],
        capture_output=True,
        text=True,
        shell=False,
    )
    selected_path = result.stdout.strip()
    if selected_path and selected_path != 'No file chosen':
        dataset_path = selected_path
        variables.last_directory = dataset_path
        print(f'Selected: {dataset_path}')

button

## 2. Load the file

The loader auto-detects the file layout: it tries the calibrated `/df`
group first and falls back to the raw `/dld` + `/tdc` groups when `/df`
isn't present. Either way, `variables.data` holds the dld dataframe and
`variables.data_tdc` holds the linked raw timestamps.

When a `/range` group exists in the file (or a sibling `<dataset>_range.h5`
is found next to it), the range table is loaded too — that enables the
*From range file* peak source in the analysis cell below.

In [ ]:
helper_data_loader.load_calibrated_h5(dataset_path, variables)
display(variables.data.head())
if variables.data_tdc is not None:
    display(variables.data_tdc.head())

## 3. Run analysis

Pick the peak source from the dropdown, fill in up to six peak windows
(only when *Manual peak windows* is selected), then click *Run analysis*.
Each section renders a plot followed by an inline Markdown summary.

- **From range file** — derives the species list from the loaded `/range`
  table; the manual rows below are disabled.
- **Manual peak windows** — type up to six `(label, mc_low, mc_up)` triples;
  rows left at 0/0 are skipped.

In [ ]:
helper_auto_raw_analysis.call_auto_raw_data_analysis(variables)